In [1]:
import os

import numpy as np
import pandas as pd

from collections import Counter

import seaborn as sns
import matplotlib.pyplot as plt

from rdkit import Chem

from routehunter_build.paper import get_work_by_doi

In [2]:
sheet_id_oprd = "18aP203JdmhgdD67P-vTjparpQqh90lN80RjOaIGSyKs"
sheet_id_other = "1p-n7Y03KD8fkRSwpkFf5hNBKzr4Kj3cMyh6YpLulnXE"
#
url_oprd = f"https://docs.google.com/spreadsheets/d/{sheet_id_oprd}/export?format=csv"
url_other = f"https://docs.google.com/spreadsheets/d/{sheet_id_other}/export?format=csv"

### 1. Collection progress

In [ ]:
tmp = pd.concat([pd.read_csv(url_oprd), pd.read_csv(url_other)])
tot = len(tmp)
print(f"Total papers:     {tot}")
tmp = tmp.dropna()
proc = len(tmp)
print(f"Processed papers: {proc} ({100*proc/tot:.1f}%)")
tmp = tmp[tmp["target"] != "SKIP"]
print(f"Collected targets:   {len(tmp)}")

### 2. Check smiles

In [ ]:
target_df = pd.concat([pd.read_csv(url_oprd), pd.read_csv(url_other)]).dropna()
target_df = target_df[target_df["target"] != "SKIP"]

# smiles check
for row in target_df.iterrows():
    mol = Chem.MolFromSmiles(row[1]["target"])
    if not mol:
        raise("Invalid SMILES")
    inchikey = Chem.MolToInchiKey(mol)

### 3. Data subsets preparation

In [ ]:
api_key = "Q2bbP41q4woiJUZBbtVSpU"

In [ ]:
target_df = pd.concat([
    pd.read_csv(url_oprd).dropna(subset="target"), 
    pd.read_csv(url_other).dropna(subset="target")]
)
target_df.head()

In [ ]:
res = []
for n, (idx, paper) in enumerate(target_df.iterrows()):

    print(f"{n + 1} / {len(target_df)}", end="\r")

    # 1. Get meta data
    meta = get_work_by_doi(paper["doi"], api_key=api_key)

    # 2. Create res
    data_point = {
         
        "journal": meta["venue"],
        "title": meta["title"], 
        "abstract": meta["abstract"], 
        "year": meta["publication_year"],
        "citation": meta["cited_by_count"],
        "doi": paper["doi"],
        "target": paper["target"],
        "contributor": "Dmitry Zankov",
    }

    # 3. Get route
    if paper["target"] == "SKIP":
        data_point["has_route"] = 0
    else:
        data_point["has_route"] = 1

    # 4. Append data point
    res.append(data_point)
#
df_main = pd.DataFrame(res)

In [ ]:
# 1. Row dataset
df_main.to_csv("rh-data/core/routehunter_main.csv", index=False)

# 2. Seed dataset
df_seed = df_main.copy()[df_main["has_route"] == 1]
df_seed = df_seed.drop("has_route", axis=1)
df_seed.to_csv("rh-data/core/routehunter_seed.csv", index=False)

# 3. Route prob dataset
df_route = df_main.copy().drop_duplicates(subset="doi")
df_route = df_route[["title", "abstract", "doi", "has_route"]]
df_route = df_route.dropna(subset="title")
df_route.to_csv("rh-data/training/abstract_data.csv", index=False)

# 4. Citation prob dataset
df_cit = df_main.copy()[df_main["has_route"] == 1]
avg_cit = df_cit["citation"] / (2026 - df_cit["year"])
df_cit["avg_cit_per_year"] = avg_cit
df_cit = df_cit[["target", "doi", "avg_cit_per_year"]]
df_cit = df_cit.dropna()
df_cit.to_csv("rh-data/training/citation_data.csv", index=False)

### 4. CASP data amd model

In [ ]:
import pandas as pd
from rdkit import Chem

from sklearn.model_selection import train_test_split
from routehunter_build.casp import load_casp_data, train_casp_model, save_casp_model
from sklearn.metrics import balanced_accuracy_score

from routehunter_build.merge import merge_tool_tables

#### 4.1 Train solvability model

In [ ]:
az_data = pd.read_json("casp/aizynth_oprd.json", orient="table")[["target", "is_solved"]]
sp_data = pd.read_csv("casp/synplan_oprd/tree_search_stats.csv")[["target_smiles", "solved"]]
#
az_data.columns = ["smiles", "is_solved"]
sp_data.columns = ["smiles", "is_solved"]
#
az_data.to_csv("rh-data/training/az_data.csv", index=False)
sp_data.to_csv("rh-data/training/sp_data.csv", index=False)

##### AiZynthFinder model

In [ ]:
smiles, y = load_casp_data("rh-data/training/az_data.csv")
model = train_casp_model(smiles, y, random_state=42)
save_casp_model(model, output_path="rh-ata/model/az_model.pickle")

##### SynPlanner model

In [ ]:
smiles, y = load_casp_data("rh-data/training/sp_data.csv")
model = train_casp_model(smiles, y, random_state=42)
save_casp_model(model, output_path="rh-data/model/sp_model.pickle")

#### 4.2 Create solvability table

In [ ]:
df_merged = merge_tool_tables(
    {
     "AiZynthFinder":"rh-data/training/az_data.csv", 
     "SynPlanner": "rh-data/training/sp_data.csv"
    }
)
df_merged.to_csv("rhdata/core/routehunter_casp.csv", index=False)

### 5. Paper route probability

In [1]:
import pandas as pd

from collections import Counter
from routehunter_build.models import load_route_data, train_route_model

In [ ]:
training_data_path = "rh-data/training/abstract_data.csv"
paper_data_path = "extraction_data/openalex_chem_metadata_1.csv"
monitor_data_path = "rh-data/training/abstract_data.csv"

In [ ]:
text, y = load_route_data("rh-data/training/abstract_data.csv")
model, metrics = train_abstract_model(text_train, y_train, random_state=42)
y_prob = model.predict_proba(text_val)[:, 1]

In [ ]:
target_precision = 0.8

precisions, recalls, thresholds = precision_recall_curve(y_val, y_prob)
idx = next(i for i, p in enumerate(precisions) if p >= target_precision)
best_threshold = thresholds[idx]

print(f"Best threshold: {best_threshold:.3f}, Precision: {precisions[idx]:.3f}, Recall: {recalls[idx]:.3f}")

In [ ]:
df = pd.read_csv("openalex_chem_metadata_final.csv")
#
text = [combine_text(t, a) for t, a in zip(df["title"], df["abstract"])]
df["route_prob"] = model.predict_proba(text)[:, 1]
print(f"Total number of papers: {len(df)}")

In [ ]:
df = df[["journal", "title", "abstract", "doi", "publication_date", "route_prob"]]
df = df[df["route_prob"] > best_threshold]
print(f"Number of papers with route_prob > {best_threshold}: {len(df)}")
df.to_csv("rh-data/monitor/paper_route_prob_medium.csv", index=False)

In [ ]:
Counter(df["journal"])

In [ ]:
df.sort_values(by="route_prob", ascending=False)

In [ ]:
jour = 'Green Chemistry'
df[df["journal"] == jour].sort_values(by="route_prob", ascending=False)

In [ ]:
JOURNAL_NAMES = [
    'Journal of the American Chemical Society',    
]


In [ ]:
idx = 637345

print(df.loc[idx].title)
print(df.loc[idx].doi)

In [ ]:
tmp = df[df["journal"] == 'Journal of the American Chemical Society'].sort_values(by="route_prob", ascending=False).head(20)
for i, row in tmp.iterrows():
    print(row.title)
    print(row.doi)
    print(row.route_prob)
    print()

In [ ]:
df.sort_values(by="route_prob", ascending=False)

### 6. Route citation prediction

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

from routehunter_build.citation import load_citation_data, train_citation_model, save_citation_model

In [ ]:
smiles, y = load_citation_data("rh-data/training/citation_data.csv")
model = train_citation_model(smiles_train, y_train, random_state=42)
save_citation_model(model, output_path="rh-ata/model/citation_model.pickle")